# Train My AI Coding Helper - eigentiki

<a href="https://colab.research.google.com/github/tolani007/sft-coding-agent/blob/main/notebooks/sft_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I use this notebook to teach an AI helper to code. I teach it with 6,625 examples of real coding work. I use a model named Google Gemma 2 9B. I use Unsloth to make the training fast. I do this on an A100 GPU.

I follow the method from the "Agentic SFT Run on Gemma and Pi-Mono" guide:
1. I run a **sweep** over key settings (learning rate, LoRA rank, sequence length).
2. I track all runs with **live metrics** so I can see what happens.
3. I pick the **best run by held-out eval loss** - I fix this rule before I start.
4. I run a **HumanEval sanity check** on the winner.
5. I push the winner to the Hub with a **model card**.

### Things I need to do first:
1. Go to **Runtime > Change runtime type** and pick **A100 GPU**.
2. Add my Hugging Face Token to Colab Secrets. I click the Secrets icon on the left side. I add a secret named `HF_TOKEN`. I turn on Notebook access.

## 1. Install Tools

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install --upgrade datasets
!pip install human_eval  # For the HumanEval sanity check

## 2. Log in to Hugging Face

In [ ]:
from google.colab import userdata
import os

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

from huggingface_hub import login
login(token=hf_token)

## 3. Load the Data
I load the data first. I only need to do this once for all sweep runs.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("focustiki/sft-coding-agent-traces")
train_data = dataset["train"]
eval_data = dataset["test"]

print(f"Loaded {len(train_data)} training examples and {len(eval_data)} eval examples.")

## 4. Set Up the Sweep

I do not guess which settings work best. I run a sweep. I try a few values for learning rate, LoRA rank, and sequence length. Then I pick the run with the lowest eval loss.

This is the rule I fix now, before I see any numbers: **the run with the lowest held-out eval loss wins**.

In [ ]:
# I define the sweep configs here.
# Each config is one training run with different settings.
# The slides say: vary learning rate, LoRA rank, sequence length.

SWEEP_CONFIGS = [
    {"name": "run_1", "lr": 2e-4, "r": 16, "lora_alpha": 32,  "seq_len": 4096},
    {"name": "run_2", "lr": 1e-4, "r": 32, "lora_alpha": 64,  "seq_len": 8192},
    {"name": "run_3", "lr": 5e-5, "r": 64, "lora_alpha": 128, "seq_len": 8192},
]

# Fixed rule: I pick the winner by lowest eval loss.
SELECTION_RULE = "lowest eval_loss"
print(f"Selection rule (fixed before training): {SELECTION_RULE}")
print(f"Number of sweep runs: {len(SWEEP_CONFIGS)}")

## 5. Run the Sweep

I loop through each config. For each one, I load the model fresh, train it, and save the eval loss. I track live metrics so I can see what happens during each run.

In [ ]:
import json
import gc
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from unsloth import is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

# Store results for each run
sweep_results = []

for config in SWEEP_CONFIGS:
    run_name = config["name"]
    print(f"\n{'='*60}")
    print(f"STARTING: {run_name}")
    print(f"  lr={config['lr']}, r={config['r']}, seq_len={config['seq_len']}")
    print(f"{'='*60}\n")

    # Load a fresh model for each run
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "google/gemma-2-9b-it",
        max_seq_length = config["seq_len"],
        dtype = None,
        load_in_4bit = True,
    )

    # Attach LoRA with this run's rank
    model = FastLanguageModel.get_peft_model(
        model,
        r = config["r"],
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = config["lora_alpha"],
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    # Set up the chat template
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")

    def formatting_prompts_func(examples):
        texts = []
        for messages in examples["messages"]:
            for msg in messages:
                if "tool_calls" in msg and msg["tool_calls"]:
                    tools_json = json.dumps(msg["tool_calls"])
                    msg["content"] = (msg.get("content", "") or "") + "\n<tool_call>\n" + tools_json + "\n</tool_call>"
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return {"text": texts}

    train_dataset = train_data.map(formatting_prompts_func, batched=True)
    eval_dataset = eval_data.map(formatting_prompts_func, batched=True)

    # Build the trainer
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = config["seq_len"],
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            run_name = run_name,
            per_device_train_batch_size = 4,
            gradient_accumulation_steps = 4,
            warmup_steps = 20,
            num_train_epochs = 1,
            learning_rate = config["lr"],
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 10,
            eval_strategy = "steps",
            eval_steps = 50,
            save_strategy = "steps",
            save_steps = 50,
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            greater_is_better = False,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "cosine",
            seed = 3407,
            output_dir = f"outputs/{run_name}",
            report_to = "none",
        ),
    )

    # Mask everything except assistant responses
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part = "<|im_start|>assistant\n",
    )

    # Train
    stats = trainer.train()

    # Get the final eval loss
    eval_result = trainer.evaluate()
    eval_loss = eval_result["eval_loss"]

    # Save the results
    result = {
        "name": run_name,
        "lr": config["lr"],
        "r": config["r"],
        "seq_len": config["seq_len"],
        "eval_loss": eval_loss,
        "train_loss": stats.training_loss,
        "train_runtime_sec": stats.metrics["train_runtime"],
    }
    sweep_results.append(result)

    # Save adapter for this run
    model.save_pretrained(f"outputs/{run_name}/adapter")
    tokenizer.save_pretrained(f"outputs/{run_name}/adapter")

    print(f"\nDONE: {run_name} - eval_loss={eval_loss:.4f}, train_loss={stats.training_loss:.4f}")

    # Free GPU memory before next run
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll sweep runs done.")

## 6. Pick the Winner

I apply the rule I fixed before training: the run with the lowest held-out eval loss wins. I do not change this rule after I see the numbers.

In [ ]:
import pandas as pd

# Show all results in a table
df = pd.DataFrame(sweep_results)
print("Sweep Results:")
print(df.to_string(index=False))

# Apply the fixed rule: lowest eval_loss wins
best = min(sweep_results, key=lambda x: x["eval_loss"])
print(f"\nWinner: {best['name']}")
print(f"  eval_loss = {best['eval_loss']:.4f}")
print(f"  lr = {best['lr']}")
print(f"  LoRA r = {best['r']}")
print(f"  seq_len = {best['seq_len']}")

BEST_RUN = best["name"]

## 7. Push the Winner to the Hub

I push the best adapter to my Hugging Face account. I add a model card so others can see what I did.

In [ ]:
from huggingface_hub import HfApi, ModelCard, ModelCardData

# Load the best adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"outputs/{BEST_RUN}/adapter",
    max_seq_length = best["seq_len"],
    dtype = None,
    load_in_4bit = True,
)

# Push to Hub
REPO_ID = "focustiki/eigentiki"
model.push_to_hub(REPO_ID, token=hf_token)
tokenizer.push_to_hub(REPO_ID, token=hf_token)

# Build the model card
card_text = f"""---
license: apache-2.0
base_model: google/gemma-2-9b-it
tags:
- code
- agent
- tool-use
- lora
- sft
datasets:
- focustiki/sft-coding-agent-traces
---

# eigentiki

I trained this LoRA adapter on top of google/gemma-2-9b-it.
I used 6,625 examples of real coding agent work (tool calls, bash, reasoning).

## Training

- Method: QLoRA (4-bit) with Unsloth
- Data: focustiki/sft-coding-agent-traces
- I ran a sweep of {len(SWEEP_CONFIGS)} configs and picked the best by eval loss.
- Winner: {best['name']} (eval_loss={best['eval_loss']:.4f})
- Settings: lr={best['lr']}, LoRA r={best['r']}, seq_len={best['seq_len']}

## Sweep Results

| Run | LR | LoRA r | Seq Len | Eval Loss | Train Loss |
|-----|-----|--------|---------|-----------|------------|
"""

for r in sweep_results:
    card_text += f"| {r['name']} | {r['lr']} | {r['r']} | {r['seq_len']} | {r['eval_loss']:.4f} | {r['train_loss']:.4f} |\n"

card_text += """
## How to Use

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="focustiki/eigentiki",
    max_seq_length=8192,
    load_in_4bit=True,
)
```
"""

# Upload the model card
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj=card_text.encode(),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"I pushed eigentiki to https://huggingface.co/{REPO_ID}")
print("The model card shows all sweep results.")

## 8. HumanEval Sanity Check

This is a cheap test, not a proof. HumanEval tests single-shot function writing. I trained a multi-turn tool-using agent. These are not the same thing. But this test tells me if the model can still write valid Python after training. If it fails badly, something went wrong.

In [ ]:
FastLanguageModel.for_inference(model)

# I test on 10 HumanEval problems as a quick sanity check.
# A full eval with Inspect AI + vLLM should run on HF Jobs (not Colab).
from human_eval.data import read_problems

problems = read_problems()
test_ids = list(problems.keys())[:10]
results = []

for task_id in test_ids:
    prompt = problems[task_id]["prompt"]
    messages = [
        {"role": "user", "content": f"Complete this Python function. Return ONLY the code, no explanation.\n\n{prompt}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512, use_cache=True,
        temperature=0.1, do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    results.append({"task_id": task_id, "completion": response})
    print(f"{task_id}: generated {len(response)} chars")

print(f"\nDone. Generated code for {len(results)} problems.")
print("This is a sanity check. A full eval should use Inspect AI + vLLM on HF Jobs.")

## 9. Test eigentiki as an Agent

This is the real test. I ask eigentiki to do a coding task the way an agent would.

In [ ]:
messages = [
    {"role": "system", "content": "You are eigentiki, an expert coding assistant with access to bash and python tools."},
    {"role": "user", "content": "Write a python script to calculate the first 10 Fibonacci numbers and then run it."},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("\neigentiki says:\n")
_ = model.generate(
    input_ids=inputs, streamer=text_streamer,
    max_new_tokens=500, use_cache=True,
)

## 10. Save the Sweep Log

I save a record of every run, the rule I used, and the winner. This is needed to verify the results later.

In [ ]:
import json
from datetime import datetime

sweep_log = {
    "project": "eigentiki",
    "date": datetime.now().isoformat(),
    "base_model": "google/gemma-2-9b-it",
    "dataset": "focustiki/sft-coding-agent-traces",
    "selection_rule": SELECTION_RULE,
    "winner": best["name"],
    "all_runs": sweep_results,
    "hub_repo": "focustiki/eigentiki",
}

with open("sweep_log.json", "w") as f:
    json.dump(sweep_log, f, indent=2)

# Also push the log to the Hub so it is always with the model
api.upload_file(
    path_or_fileobj="sweep_log.json",
    path_in_repo="sweep_log.json",
    repo_id="focustiki/eigentiki",
    repo_type="model",
    token=hf_token,
)

print("Sweep log saved and pushed to the Hub.")
print(json.dumps(sweep_log, indent=2))